# 课后练习解答（02.05_training_pipeline）

本解答对应章节课后练习，共 15 题。

### 问题1（单选题）

**题目：** CosineAnnealingLR(T_max=50, eta_min=0) 中 T_max 的含义是？
A. 完成半个余弦周期的 epoch 数
B. 总训练 epoch 数
C. 初始学习率
D. 最小学习率

**解答：** A

**解析：** PyTorch 的 CosineAnnealingLR 在 T_max 个 epoch 内从初始 LR 衰减到 eta_min，之后进入新的余弦周期。


### 问题2（单选题）

**题目：** 200 类分类任务中，标签平滑 epsilon=0.1 时，错误类别的目标值是多少？
A. 0.1/199
B. 0.9/199
C. 0.1/200
D. 0

**解答：** A

**解析：** 正确类别目标为 1-epsilon=0.9，其余 0.1 平均分给 199 个错误类别。


### 问题3（单选题）

**题目：** EMA 更新公式 shadow = decay*shadow + (1-decay)*param 中，decay=0.999 的作用是？
A. 更看重近期参数
B. 更看重长期历史参数
C. 与历史无关
D. 等价于学习率

**解答：** B

**解析：** decay 越接近 1，历史影子参数占比越高，滑动平均越平滑。


### 问题4（多选题）

**题目：** 将 batch_size 从 64 提升到 256，通常需要同步调整？
A. 学习率
B. warmup 步数
C. 梯度裁剪阈值
D. 数据增强策略

**解答：** AB

**解析：** 大 batch 通常按线性缩放学习率，并适当延长 warmup 稳定训练；裁剪阈值与增强策略一般无需联动修改。


### 问题5（多选题）

**题目：** 一个规范的 Warmup + Cosine 学习率计划包含？
A. 线性/常数 warmup 阶段
B. 余弦衰减阶段
C. 最小学习率
D. 总训练步数

**解答：** ABCD

**解析：** 四者共同构成完整调度：warmup 稳定起步，cosine 平滑衰减，eta_min 与总步数决定衰减曲线。


### 问题6（判断题）

**题目：** RMSprop 通过除以梯度平方的滑动平均来归一化每个参数的学习步长。

**解答：** 对

**解析：** RMSprop 维护梯度平方的 EMA，并用其平方根归一化梯度，缓解梯度尺度差异。


### 问题7（判断题）

**题目：** EMA 影子参数应在 optimizer.step() 之前更新，才能包含本轮梯度。

**解答：** 错

**解析：** 必须先在 optimizer.step() 更新模型参数，再更新影子参数，否则影子参数不包含本轮参数变化。


### 问题8（填空题）

**题目：** model.eval() 会关闭 ____ 层，并让 BatchNorm 使用 ____ 统计量。

**解答：** Dropout；running（滑动）


### 问题9（填空题）

**题目：** 在昇腾 NPU 上使用混合精度训练时，防止梯度下溢通常使用 ____ 类做梯度缩放。

**解答：** GradScaler（torch.npu.amp.GradScaler）


### 问题10（简答题）

**题目：** 标签平滑为什么能抑制模型过度自信？请从 softmax 输出和目标分布角度说明。

**解答：** 硬标签会让模型把正确类别概率推向 1，logits 无限增大；标签平滑把目标分布改为 1-epsilon 与 epsilon/(C-1)，给 logits 设置有限边界，使输出概率不过度集中，提升校准性与泛化性。


### 问题11（简答题）

**题目：** 为什么线性 warmup 能降低大学习率带来的早期不稳定性？

**解答：** 训练初期梯度方差大、参数尚未适应损失曲面，直接使用大学习率容易发散；warmup 从小学习率逐步过渡，等价于给优化器一段稳定预热期。


### 问题12（代码设计题）

**题目：** 写出 train_one_step 骨架：forward → loss → zero_grad → backward → 可选 grad clip → step → EMA 更新。

**解答：** ```python
from contextlib import nullcontext
def train_one_step(model, batch, criterion, optimizer, scaler=None, max_grad_norm=None, ema=None):
    images, labels = batch
    images, labels = images.to(device), labels.to(device)
    optimizer.zero_grad()
    with torch.npu.amp.autocast() if scaler else nullcontext():
        loss = criterion(model(images), labels)
    if scaler:
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        if max_grad_norm: torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        scaler.step(optimizer)
        scaler.update()
    else:
        loss.backward()
        if max_grad_norm: torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()
    if ema: ema.update(model)
    return loss.item()
```


### 问题13（单选题）

**题目：** 训练 loss 持续下降，验证准确率连续 8 个 epoch 不提升，最可能的判断是？
A. 过拟合
B. 欠拟合
C. 学习率过高
D. 数据标签错误

**解答：** A

**解析：** 训练指标继续改善而验证不再提升，是典型的过拟合信号。


### 问题14（多选题）

**题目：** 训练日志出现以下哪些信号时，应暂停训练并排查稳定性？
A. loss 出现 NaN
B. 梯度范数指数级增大
C. 验证 loss 剧烈震荡
D. 训练 acc 单调上升

**解答：** ABC

**解析：** NaN、梯度爆炸、验证震荡都指向不稳定；训练 acc 单调上升是正常现象。


### 问题15（简答题）

**题目：** 设计一个 CPU/NPU 公平对比实验，说明需要固定的变量与测量方法。

**解答：** 固定相同模型、权重、输入 shape、dtype、batch_size、warmup 与 repeats；CPU 侧用 time.perf_counter，NPU 侧在计时后调用 synchronize；分别取多次测量均值或中位数，并记录硬件与软件版本。
